# M21C Phase 1 trends and observing-system transitions

Audited domain-mean summary of known-date interrupted-series estimates and independently detected changepoints. Independent dates require penalty stability and agreement between AR+trend and prewhitened PELT methods.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import xarray as xr

REPO = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'projects' / 'M21C_ls').exists())
PROJECT = REPO / 'projects' / 'M21C_ls'
OUTPUT = PROJECT / 'output' / 'trends_breakpoints'
coefficients = pd.read_csv(OUTPUT / 'phase1_interrupted_series_coefficients.csv')
detections = pd.read_csv(OUTPUT / 'phase1_changepoint_detections.csv', parse_dates=['break_date'])
comparison = pd.read_csv(OUTPUT / 'phase1_changepoint_boundary_comparison.csv', parse_dates=['boundary_date', 'detected_date'])
penalty_grid = pd.read_csv(OUTPUT / 'phase1_changepoint_penalty_grid.csv', parse_dates=['break_date'])
with xr.open_dataset(OUTPUT / 'phase1_changepoint_monthly.nc') as source:
    monthly = source.load()
registry = json.loads((PROJECT / 'config' / 'observing_system_registry.json').read_text())
periods = pd.DataFrame(registry['fine_periods'])
periods['start'] = pd.to_datetime(periods['start'])
accepted = detections[detections['accepted_detection']].copy()

plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'figure.dpi': 120})
COLORS = {'OL': '#666666', 'DA': '#2878B5', 'delta': '#C23B33', 'P6': '#8B1E1E'}

def panel_labels(axes):
    for index, ax in enumerate(np.ravel(axes)):
        ax.text(0.01, 0.98, f'({chr(97 + index)})', transform=ax.transAxes, va='top', ha='left', fontweight='bold')

def add_boundaries(ax, emphasize='P6'):
    for row in periods.iloc[1:].itertuples():
        ax.axvline(row.start, color=COLORS['P6'] if row.period_id == emphasize else '#B7B7B7', lw=1.5 if row.period_id == emphasize else 0.7, alpha=0.9)

def standardized(series_id):
    values = monthly['seasonal_adjusted'].sel(series=series_id).values.astype(float)
    return (values - np.nanmean(values)) / np.nanstd(values, ddof=1)

print(f"Accepted independent breaks: {len(accepted)}; controls: {(accepted['series_role'] == 'paired_control').sum()}")

## Figure 1: P6 convergence in soil-water impacts

Seasonally adjusted series are standardized separately so variables with different units can be compared. The dark vertical line is April 2015, when SMAP assimilation begins; coherent steps at that line support a P6 transition.

In [ ]:
groups = [
    ('RZMC DA - OL', [
        ('RZMC_delta_valid_land__delta', 'valid land'),
        ('RZMC_delta_warm_snowfree_monthly__delta', 'warm snow-free'),
    ]),
    ('Soil-water correction activity', [
        ('soil_water_net_approx_value_valid_land__value', 'signed net'),
        ('soil_water_abs_activity_value_valid_land__value', 'absolute activity'),
    ]),
    ('Soil-moisture correction magnitude', [
        ('SFMC_INC_ABS_MEAN_value_valid_land__value', 'SFMC absolute mean'),
        ('SFMC_INC_RMS_value_valid_land__value', 'SFMC RMS'),
        ('RZMC_INC_ABS_MEAN_value_valid_land__value', 'RZMC absolute mean'),
        ('RZMC_INC_RMS_value_valid_land__value', 'RZMC RMS'),
    ]),
]
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, constrained_layout=True)
for ax, (title, series_group) in zip(axes, groups):
    for series_id, label in series_group:
        ax.plot(monthly.time.values, standardized(series_id), lw=1.3, label=label)
    add_boundaries(ax)
    ax.axhline(0, color='#888888', lw=0.6)
    ax.set_title(title)
    ax.set_ylabel('Standardized value')
    ax.legend(ncol=2, frameon=False, loc='upper left', bbox_to_anchor=(0.04, 0.98))
axes[-1].set_xlabel('Year')
panel_labels(axes)
fig.suptitle('Soil-water impacts around the P6 / SMAP transition', fontsize=14)
plt.show()

## Figure 2: known-date P6 level changes

Points and bars are P6 level estimates and 95% bootstrap intervals. Positive values indicate an upward shift after SMAP entry. Red points survive boundary-family FDR; grey points do not.

In [ ]:
p6_ids = accepted.loc[accepted['break_date'] == pd.Timestamp('2015-04-01'), 'series_id'].unique()
p6 = coefficients[(coefficients['coefficient'] == 'level_change_P6') & coefficients['series_id'].isin(p6_ids)].copy()
label_map = {
    'RZMC_delta_valid_land__delta': 'RZMC DA - OL, valid land',
    'RZMC_delta_warm_snowfree_monthly__delta': 'RZMC DA - OL, warm snow-free',
    'SFMC_INC_MEAN_value_valid_land__value': 'SFMC increment mean',
    'SFMC_INC_ABS_MEAN_value_valid_land__value': 'SFMC increment absolute mean',
    'SFMC_INC_RMS_value_valid_land__value': 'SFMC increment RMS',
    'RZMC_INC_MEAN_value_valid_land__value': 'RZMC increment mean',
    'RZMC_INC_ABS_MEAN_value_valid_land__value': 'RZMC increment absolute mean',
    'RZMC_INC_RMS_value_valid_land__value': 'RZMC increment RMS',
    'soil_water_net_approx_value_valid_land__value': 'Soil-water signed net correction',
    'soil_water_abs_activity_value_valid_land__value': 'Soil-water absolute activity',
}
p6['display'] = p6['series_id'].map(label_map)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), constrained_layout=True)
for ax, (units, frame) in zip(axes, p6.groupby('estimate_units', sort=False)):
    frame = frame.sort_values('estimate').reset_index(drop=True)
    y = np.arange(len(frame))
    colors = np.where(frame['significant_fdr'], '#C23B33', '#777777')
    ax.hlines(y, frame['ci_low_bootstrap'], frame['ci_high_bootstrap'], color=colors, lw=2)
    ax.scatter(frame['estimate'], y, c=colors, s=42, zorder=3)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(y, frame['display'])
    ax.set_xlabel(f'P6 level change ({units})')
    ax.grid(axis='x', alpha=0.2)
panel_labels(axes)
fig.suptitle('Known-date P6 level changes for independently detected April 2015 series', fontsize=13)
plt.show()

## Figure 3: independent-date agreement with P2-P9

Cells show detected month minus known boundary month. Blue is early, red is late, and white is exact; grey means no match within six months. P7 is hatched because it is detection-exempt. Color direction is timing, not better or worse.

In [ ]:
primary = comparison[comparison['series_role'] == 'primary_estimand'].copy()
series_order = primary[['series_id', 'variable', 'mask']].drop_duplicates()['series_id'].tolist()
period_order = [f'P{i}' for i in range(2, 10)]
matrix = np.full((len(series_order), len(period_order)), np.nan)
for i, series_id in enumerate(series_order):
    for j, period_id in enumerate(period_order):
        row = primary[(primary['series_id'] == series_id) & (primary['period_id'] == period_id)].iloc[0]
        if row['matched_within_sensitivity_tolerance']:
            matrix[i, j] = row['signed_offset_months']
labels = [f"{primary.loc[primary['series_id'] == sid, 'variable'].iloc[0]} | {primary.loc[primary['series_id'] == sid, 'mask'].iloc[0]}" for sid in series_order]
cmap = ListedColormap(['#31688E', '#8CC5D6', '#FFFFFF', '#F3A481', '#B33A3A'])
cmap.set_bad('#D8D8D8')
norm = BoundaryNorm([-6.5, -3.5, -0.5, 0.5, 3.5, 6.5], cmap.N)
fig, ax = plt.subplots(figsize=(10, 10), constrained_layout=True)
mesh = ax.pcolormesh(np.ma.masked_invalid(matrix), cmap=cmap, norm=norm, edgecolors='white', linewidth=0.6)
ax.set_xticks(np.arange(len(period_order)) + 0.5, period_order)
ax.set_yticks(np.arange(len(series_order)) + 0.5, labels)
ax.invert_yaxis()
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        if np.isfinite(matrix[i, j]):
            ax.text(j + 0.5, i + 0.5, f'{int(matrix[i, j]):+d}', ha='center', va='center', fontsize=8)
p7_index = period_order.index('P7')
ax.add_patch(Rectangle((p7_index, 0), 1, len(series_order), fill=False, hatch='////', edgecolor='#555555', lw=1.0))
cbar = fig.colorbar(mesh, ax=ax, orientation='horizontal', pad=0.04, fraction=0.045, ticks=[-5, -2, 0, 2, 5])
cbar.set_label('Detected minus known boundary (months)')
ax.set_title('Independent changepoint agreement for primary Phase 1 estimands')
plt.show()

## Figure 4: OL, DA, and DA-minus-OL controls

Each line is standardized separately. Grey and blue show the paired OL and DA state series; red shows DA minus OL. Red dotted lines mark accepted breaks in the delta. Accepted control breaks would be drawn in their line color, but none occur.

In [ ]:
runs = [
    ('SFMC_delta_valid_land', 'SFMC, valid land'),
    ('RZMC_delta_valid_land', 'RZMC, valid land'),
    ('SFMC_delta_warm_snowfree_monthly', 'SFMC, warm snow-free'),
    ('RZMC_delta_warm_snowfree_monthly', 'RZMC, warm snow-free'),
]
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, constrained_layout=True)
for ax, (run_id, title) in zip(axes.ravel(), runs):
    for source_series, label in [('ol', 'OL'), ('da', 'DA'), ('delta', 'DA - OL')]:
        series_id = f'{run_id}__{source_series}'
        ax.plot(monthly.time.values, standardized(series_id), color=COLORS.get(label, COLORS.get(source_series)), lw=1.0, label=label)
        dates = accepted.loc[accepted['series_id'] == series_id, 'break_date']
        for date in dates:
            ax.axvline(date, color=COLORS.get(label, COLORS.get(source_series)), lw=1.2, ls=':')
    add_boundaries(ax)
    ax.set_title(title)
    ax.set_ylabel('Standardized value')
    ax.axhline(0, color='#888888', lw=0.5)
axes[0, 0].legend(frameon=False, ncol=3, loc='upper left', bbox_to_anchor=(0.06, 0.98))
for ax in axes[-1]:
    ax.set_xlabel('Year')
panel_labels(axes)
fig.suptitle('Paired state controls and assimilation-impact series', fontsize=14)
plt.show()

## Figure 5: penalty sensitivity and accepted-break stability

The left panel shows how detected-break counts decline as penalties become stricter. The right panel shows stability across the six penalties for accepted breaks; higher values indicate stronger penalty robustness.

In [ ]:
counts = penalty_grid.groupby(['method', 'penalty_multiplier']).size().rename('n').reset_index()
method_labels = {'pelt_ar1_linear': 'AR(1) + linear', 'pelt_prewhitened_linear': 'Prewhitened linear'}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for method, frame in counts.groupby('method'):
    axes[0].plot(frame['penalty_multiplier'], frame['n'], marker='o', label=method_labels[method])
axes[0].axvline(1.25, color='black', ls=':', lw=1.0, label='primary penalty')
axes[0].set_xlabel('BIC penalty multiplier')
axes[0].set_ylabel('Detected breaks across 43 series')
axes[0].legend(frameon=False)
bins = np.arange(0.45, 1.05, 0.1)
axes[1].hist(accepted['penalty_stability_fraction'], bins=bins, color='#2878B5', edgecolor='white')
axes[1].set_xlabel('Penalty stability fraction')
axes[1].set_ylabel('Accepted breaks')
axes[1].set_xticks([0.5, 2/3, 5/6, 1.0], ['0.50', '0.67', '0.83', '1.00'])
panel_labels(axes)
fig.suptitle('Independent changepoint penalty diagnostics', fontsize=13)
plt.show()